In [1]:
!pip install faiss-cpu sentence-transformers openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 24.6 MB/s eta 0:00:00


In [50]:
# @title
!pip install faiss-cpu sentence-transformers google-genai

In [52]:
from sentence_transformers import SentenceTransformer
import faiss
from google import genai

In [ ]:
client = genai.Client(api_key="xxxxxxxxxxxxxxxxxxxxxxx")

In [38]:
dcs = [
    """Cristiano Ronaldo | Early Life | Cristiano Ronaldo dos Santos Aveiro was born on February 5, 1985, in Funchal, Madeira, Portugal. He grew up in a working-class family and showed exceptional football talent from a young age. At age 12, he moved away from home to join Sporting CP's youth academy in Lisbon. This move marked the beginning of his professional football journey and required significant personal sacrifice and adaptation.""",

    """Cristiano Ronaldo | Health | At age 15, Cristiano Ronaldo was diagnosed with tachycardia, a heart condition that caused his heart to race abnormally. The condition threatened his football career and required corrective surgery. The operation was successful, and after a short recovery period, Ronaldo returned to training and continued pursuing professional football. This challenge is often cited as an example of his resilience and determination.""",

    """Cristiano Ronaldo | Sporting CP | Ronaldo progressed through Sporting CP's youth system and made his first-team debut during the 2002–03 season. His speed, dribbling ability, and technical skills attracted attention from major European clubs. A standout performance against Manchester United in a friendly match convinced Sir Alex Ferguson to sign him shortly afterward.""",

    """Cristiano Ronaldo | Manchester United | Ronaldo joined Manchester United in 2003 at the age of 18. Under Sir Alex Ferguson, he developed into one of the world's best players. During his first spell at the club, he won three Premier League titles, one UEFA Champions League title, one FIFA Club World Cup, and his first Ballon d'Or in 2008. Across both spells at Manchester United, he scored 145 goals in 346 appearances.""",

    """Cristiano Ronaldo | Real Madrid | Ronaldo transferred to Real Madrid in 2009 for a then-world-record transfer fee. During his nine seasons at the club, he became Real Madrid's all-time leading scorer with 450 goals in 438 appearances. He won four UEFA Champions League titles, two La Liga titles, and numerous domestic and international trophies. His partnership with players such as Karim Benzema and Luka Modrić contributed to one of the most successful eras in the club's history.""",

    """Cristiano Ronaldo | Juventus | Ronaldo joined Juventus in 2018. During his three seasons in Italy, he scored 101 goals in 134 appearances. He helped Juventus win two Serie A titles, one Coppa Italia, and two Supercoppa Italiana trophies. He also became the fastest Juventus player to reach several scoring milestones.""",

    """Cristiano Ronaldo | Al Nassr | Ronaldo joined Al Nassr in Saudi Arabia in late 2022. His arrival significantly increased global attention toward the Saudi Pro League. He continued maintaining an impressive goal-scoring record and became one of the league's most influential players both on and off the field.""",

    """Cristiano Ronaldo | Portugal Career | Ronaldo made his senior debut for Portugal in 2003 and became the nation's most-capped player and leading goal scorer. He captained Portugal to victory at UEFA Euro 2016, the country's first major international trophy. He also led Portugal to the UEFA Nations League title in 2019 and has represented his country in multiple FIFA World Cups and European Championships.""",

    """Cristiano Ronaldo | Playing Style | Ronaldo is known for his pace, dribbling, athleticism, aerial ability, finishing, and powerful shooting. Throughout his career, he evolved from a skillful winger into a complete forward capable of scoring with both feet and his head. His dedication to physical conditioning and continuous self-improvement has contributed to his longevity at the highest level.""",

    """Cristiano Ronaldo | Statistics | Ronaldo has scored more than 900 official senior career goals for club and country, making him one of the highest-scoring players in football history. His club statistics include 450 goals for Real Madrid, 145 goals for Manchester United, 101 goals for Juventus, and numerous goals for Sporting CP, Al Nassr, and the Portugal national team.""",

    """Cristiano Ronaldo | Awards | Ronaldo has won five Ballon d'Or awards, four European Golden Shoes, five UEFA Champions League titles, and numerous domestic league championships across England, Spain, and Italy. He has also received multiple Player of the Year awards and holds several UEFA and FIFA records.""",

    """Cristiano Ronaldo | Legacy | Ronaldo is widely regarded as one of the greatest footballers of all time. Beyond football, he built the CR7 global brand, became one of the most followed athletes on social media, and participated in various charitable initiatives and humanitarian efforts. His career is often used as an example of discipline, consistency, ambition, and professional excellence."""
]

In [39]:
model=SentenceTransformer('all-MiniLM-L6-v2')
de= model.encode(dcs).astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [40]:
de.shape

(12, 384)

In [41]:
dim=de.shape[1]

In [42]:
index= faiss.IndexFlatL2(dim)

In [43]:
index.add(de)

In [44]:

qu="What make Ronaldo special"
qe=model.encode([qu]).astype("float32")
tk=2
dist,ind=index.search(qe,tk)
rc=[dcs[i] for i in ind[0]]
print("retrived chunks")
for c in rc:
  print("-",c)

retrived chunks
- Cristiano Ronaldo | Legacy | Ronaldo is widely regarded as one of the greatest footballers of all time. Beyond football, he built the CR7 global brand, became one of the most followed athletes on social media, and participated in various charitable initiatives and humanitarian efforts. His career is often used as an example of discipline, consistency, ambition, and professional excellence.
- Cristiano Ronaldo | Playing Style | Ronaldo is known for his pace, dribbling, athleticism, aerial ability, finishing, and powerful shooting. Throughout his career, he evolved from a skillful winger into a complete forward capable of scoring with both feet and his head. His dedication to physical conditioning and continuous self-improvement has contributed to his longevity at the highest level.


In [54]:
context = "\n".join(rc)
prompt = f"""Answer the question based only on the context below.
Context: {context}
Question: {qu}
Answer:"""

print("Final Answer:\n")

# Using the lightest and fastest model: gemini-2.5-flash
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=genai.types.GenerateContentConfig(
        temperature=0.4
    )
)

print(response.text)

Final Answer:

Ronaldo is special because he is widely regarded as one of the greatest footballers of all time. Beyond football, he built the CR7 global brand, became one of the most followed athletes on social media, and participated in various charitable initiatives and humanitarian efforts. His career is often used as an example of discipline, consistency, ambition, and professional excellence.

His playing style is also special due to his pace, dribbling, athleticism, aerial ability, finishing, and powerful shooting. He evolved from a skillful winger into a complete forward capable of scoring with both feet and his head. His dedication to physical conditioning and continuous self-improvement has contributed to his longevity at the highest level.
